# PRISM OOD - CLIP (Embedding-Based, Fast)
## Uses pre-saved embeddings - no GPU needed!
### 4 pairs: PCam↔MHIST | CRC↔BRACS (binarized)

In [1]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from google.colab import drive
drive.mount('/content/drive')

EMB_DIR     = '/content/drive/MyDrive/PRISM/embeddings/clip'
RESULTS_DIR = '/content/drive/MyDrive/PRISM/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 123, 456]
MKEY = 'clip'

# CRC class order (sorted folder names in NCT-CRC-HE-100K):
# ADI=0, BACK=1, DEB=2, LYM=3, MUC=4, MUS=5, NORM=6, STR=7, TUM=8
CRC_BINARY = lambda y: (y == 8).astype(int)   # TUM=1, rest=0

# BRACS class order (sorted folder names):
# ADH=0, DCIS=1, FEA=2, IC=3, N=4, PB=5, UDH=6
BRACS_BINARY = lambda y: np.isin(y, [1, 3]).astype(int)  # DCIS=1, IC=1, rest=0

print(f"Model: CLIP")
print(f"Embedding dir: {EMB_DIR}")
print("No GPU needed - using pre-saved embeddings!")

Mounted at /content/drive
Model: CLIP
Embedding dir: /content/drive/MyDrive/PRISM/embeddings/clip
No GPU needed - using pre-saved embeddings!


In [2]:
# Load all embeddings from Drive
def load_emb(dataset, split):
    path = f'{EMB_DIR}/{dataset}'
    feats  = np.load(f'{path}/{split}_features.npy')
    labels = np.load(f'{path}/{split}_labels.npy')
    return feats, labels

print("Loading embeddings...")

# Binary datasets (already binary labels: 0/1)
pcam_train_X,  pcam_train_y  = load_emb('pcam',  'train')
pcam_test_X,   pcam_test_y   = load_emb('pcam',  'test')
mhist_train_X, mhist_train_y = load_emb('mhist', 'train')
mhist_test_X,  mhist_test_y  = load_emb('mhist', 'test')

# Multiclass datasets -> binarize labels
crc_train_X,   _crc_train_y  = load_emb('crc',   'train')
crc_test_X,    _crc_test_y   = load_emb('crc',   'test')
crc_train_y  = CRC_BINARY(_crc_train_y)
crc_test_y   = CRC_BINARY(_crc_test_y)

bracs_train_X, _bracs_train_y = load_emb('bracs', 'train')
bracs_test_X,  _bracs_test_y  = load_emb('bracs', 'test')
bracs_train_y = BRACS_BINARY(_bracs_train_y)
bracs_test_y  = BRACS_BINARY(_bracs_test_y)

print(f"PCam train: {pcam_train_X.shape} | test: {pcam_test_X.shape}")
print(f"MHIST train: {mhist_train_X.shape} | test: {mhist_test_X.shape}")
print(f"CRC binary train: {crc_train_X.shape} dist={np.bincount(crc_train_y)}")
print(f"BRACS binary train: {bracs_train_X.shape} dist={np.bincount(bracs_train_y)}")

Loading embeddings...
PCam train: (262144, 768) | test: (32768, 768)
MHIST train: (1849, 768) | test: (977, 768)
CRC binary train: (70000, 768) dist=[59882 10118]
BRACS binary train: (3657, 768) dist=[2319 1338]


In [3]:
def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(labels[mask].mean() - probs[mask].mean())
    return float(ece)

def temperature_scale_binary(logits, labels):
    def nll(T):
        p = np.clip(1 / (1 + np.exp(-logits / T)), 1e-7, 1-1e-7)
        return -np.mean(labels * np.log(p) + (1-labels) * np.log(1-p))
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def stratified_sample(n_total, labels, fraction, seed):
    np.random.seed(seed)
    indices = np.arange(n_total)
    classes = np.unique(labels)
    sampled = []
    for c in classes:
        c_idx = indices[labels == c]
        n = max(1, int(len(c_idx) * fraction))
        sampled.extend(np.random.choice(c_idx, size=n, replace=False))
    return np.array(sampled)

print("Helper functions ready!")

Helper functions ready!


In [4]:
def run_ood(src_X, src_y, src_name, tgt_X, tgt_y, tgt_name):
    print(f"\n{'='*55}")
    print(f"OOD: {src_name} -> {tgt_name}")
    print(f"Src: {src_X.shape} | Tgt: {tgt_X.shape}")
    print(f"Src dist: {np.bincount(src_y.astype(int))} | Tgt dist: {np.bincount(tgt_y.astype(int))}")
    print(f"{'='*55}")

    results, ts_results = [], []

    for frac in LABEL_FRACTIONS:
        for seed in SEEDS:
            sampled = stratified_sample(len(src_y), src_y.astype(int), frac, seed)
            X_tr, y_tr = src_X[sampled], src_y[sampled].astype(int)

            clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
            clf.fit(X_tr, y_tr)

            logits = clf.decision_function(tgt_X)
            probs  = 1 / (1 + np.exp(-logits))
            preds  = (probs > 0.5).astype(int)
            tgt_int = tgt_y.astype(int)

            try:    auroc = roc_auc_score(tgt_int, probs)
            except: auroc = float('nan')
            ece_raw = compute_ece(probs, tgt_int)
            brier   = float(brier_score_loss(tgt_int, probs))
            f1      = float(f1_score(tgt_int, preds, average='macro', zero_division=0))

            try:
                T      = temperature_scale_binary(logits, tgt_int)
                sp     = 1 / (1 + np.exp(-logits / T))
                ece_sc = compute_ece(sp, tgt_int)
                ece_imp = ece_raw - ece_sc
            except:
                T = ece_sc = ece_imp = float('nan')

            results.append({'fraction':frac,'seed':seed,'auroc':auroc,
                            'ece':ece_raw,'f1':f1,'brier':brier})
            ts_results.append({'fraction':frac,'seed':seed,'temperature':T,
                               'ece_raw':ece_raw,'ece_scaled':ece_sc,'ece_improvement':ece_imp})
            print(f"  {frac:.0%} s{seed}: AUROC={auroc:.4f} ECE={ece_raw:.4f} F1={f1:.4f}")

    tag = f"{MKEY}_ood_{src_name.lower()}_to_{tgt_name.lower()}"
    pd.DataFrame(results).to_csv(f'{RESULTS_DIR}/{tag}_results.csv', index=False)
    pd.DataFrame(ts_results).to_csv(f'{RESULTS_DIR}/{tag}_temperature_scaling.csv', index=False)
    print(f"✅ Saved: {tag}")
    return pd.DataFrame(results)

print("run_ood() ready!")

run_ood() ready!


In [5]:
# Run all 4 OOD pairs - fast, no GPU!
r1 = run_ood(pcam_train_X,  pcam_train_y,  'PCam',  mhist_test_X,  mhist_test_y,  'MHIST')
r2 = run_ood(mhist_train_X, mhist_train_y, 'MHIST', pcam_test_X,   pcam_test_y,   'PCam')
r3 = run_ood(crc_train_X,   crc_train_y,   'CRC',   bracs_test_X,  bracs_test_y,  'BRACS')
r4 = run_ood(bracs_train_X, bracs_train_y, 'BRACS', crc_test_X,    crc_test_y,    'CRC')

print("\n🎉 ALL 4 OOD PAIRS DONE!")


OOD: PCam -> MHIST
Src: (262144, 768) | Tgt: (977, 768)
Src dist: [131072 131072] | Tgt dist: [617 360]
  1% s42: AUROC=0.4690 ECE=0.2110 F1=0.4364
  1% s123: AUROC=0.4588 ECE=0.2049 F1=0.4541
  1% s456: AUROC=0.4332 ECE=0.2031 F1=0.4524
  5% s42: AUROC=0.4638 ECE=0.1953 F1=0.4696
  5% s123: AUROC=0.4661 ECE=0.2042 F1=0.4665
  5% s456: AUROC=0.4571 ECE=0.2045 F1=0.4595
  10% s42: AUROC=0.4734 ECE=0.2179 F1=0.4794
  10% s123: AUROC=0.4743 ECE=0.2215 F1=0.4784
  10% s456: AUROC=0.4728 ECE=0.1960 F1=0.4656
  25% s42: AUROC=0.4653 ECE=0.2306 F1=0.4484
  25% s123: AUROC=0.4769 ECE=0.2220 F1=0.4616
  25% s456: AUROC=0.4641 ECE=0.2371 F1=0.4493
  50% s42: AUROC=0.4667 ECE=0.2570 F1=0.4382
  50% s123: AUROC=0.4652 ECE=0.2619 F1=0.4375
  50% s456: AUROC=0.4699 ECE=0.2488 F1=0.4578
  100% s42: AUROC=0.4839 ECE=0.2518 F1=0.4658
  100% s123: AUROC=0.4839 ECE=0.2518 F1=0.4658
  100% s456: AUROC=0.4839 ECE=0.2518 F1=0.4658
✅ Saved: clip_ood_pcam_to_mhist

OOD: MHIST -> PCam
Src: (1849, 768) | Tgt: 

In [6]:
for name, r in [('PCam→MHIST',r1),('MHIST→PCam',r2),('CRC→BRACS',r3),('BRACS→CRC',r4)]:
    s = r.groupby('fraction')[['auroc','ece','f1']].mean()
    print(f"\n--- {name} (mean over 3 seeds) ---")
    print(s.round(4).to_string())


--- PCam→MHIST (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.4537  0.2063  0.4476
0.05      0.4623  0.2013  0.4652
0.10      0.4735  0.2118  0.4745
0.25      0.4688  0.2299  0.4531
0.50      0.4673  0.2559  0.4445
1.00      0.4839  0.2518  0.4658

--- MHIST→PCam (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.5762  0.2285  0.3334
0.05      0.5933  0.2407  0.3334
0.10      0.5129  0.2600  0.3334
0.25      0.5948  0.2599  0.3334
0.50      0.6141  0.2933  0.3334
1.00      0.6281  0.3117  0.3337

--- CRC→BRACS (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.5503  0.0735  0.4205
0.05      0.6071  0.1921  0.5361
0.10      0.6323  0.2125  0.5330
0.25      0.6560  0.2179  0.5557
0.50      0.6609  0.2060  0.5655
1.00      0.6608  0.2021  0.5743

--- BRACS→CRC (mean over 3 seeds) ---
           auroc     ece      f1
fraction         